## Part 4: Recycling Instruction Generation with RAG

**Objectives**
- Retrieve policy evidence using semantic embeddings and FAISS.
- Generate category-specific guidance grounded in retrieved documents.
- Test retrieval depth, relevance, and coverage across waste categories.

### 4.1 Preprocess Documents for Retrieval

This subsection validates the policy embeddings, vector index, and category coverage prepared in Part 1.

#### Code Overview: Policy retrieval pipeline validation
Validates policy document preparation by confirming successful preprocessing, embedding generation, and FAISS indexing, ensuring the retrieval system is ready for semantic search.


In [ ]:
# Validating document preparation for retrieval

print("Number of Policy Documents:")
print(len(policy_df))

print("\nIndexed Documents:")
print(policy_index.ntotal)

print("\nEmbedding Dimensions:")
print(policy_embeddings.shape)

print("\nSample Retrieval Documents:")
display(
    policy_df[
        [
            "policy_id",
            "policy_type",
            "category_text",
            "jurisdiction"
        ]
    ].head()
)

#### Code Overview: Retrieval function validation
Tests semantic search using sample recycling queries to verify accurate retrieval and ranking of relevant policy documents across waste categories.


In [ ]:
#Testing retrieval capability

sample_queries = [
    "How should glass bottles be recycled?",
    "What should I do with food waste?",
    "How should cardboard be disposed of?"
]

for query in sample_queries:

    print("=" * 70)
    print(f"Query: {query}")

    results = retrieve_policy_documents(
        query,
        top_k=2
    )

    display(results)

#### Code Overview: Policy coverage analysis

Analyzes policy coverage across waste categories to identify disparities in supporting guidance and highlight areas where retrieval may depend on limited source information.


In [ ]:
# Analyzing policy coverage

policy_coverage = (
    policy_df["category_text"]
    .str.split(", ")
    .explode()
    .value_counts()
    .reset_index()
)

policy_coverage.columns = [
    "Category",
    "Number_of_Policies"
]

display(policy_coverage)

plt.figure(figsize=(10, 6))

sns.barplot(
    data=policy_coverage,
    x="Category",
    y="Number_of_Policies",
    color="#0B6E4F"
)

plt.title("Policy Coverage by Waste Category")
plt.xlabel("Waste Category")
plt.ylabel("Number of Policies")

plt.xticks(rotation=35)

plt.tight_layout()
plt.show()

#### RAG Retrieval performance summary

All 14 policy documents were successfully embedded and indexed for semantic retrieval, enabling relevant document retrieval through natural-language queries. While retrieval performance was effective across waste categories, variations in policy coverage indicate that retrieval scores should be interpreted alongside source documents rather than as indicators of equal policy depth

### 4.2 Implement RAG-based System

The system combines semantic retrieval with a lightweight policy-grounded response generator to prioritize factual consistency and local policy evidence.

#### Code Overview: RAG response generation setup
Marks the transition to the response-generation stage, confirming completion of retrieval preparation and readiness to build retrieval and generation functions.


In [ ]:

# Simple RAG response generator

print("RAG response generator initialized successfully.")

#### Code Overview: Policy retrieval mechanism

Retrieves and combines the most relevant policy guidance for a waste category, providing grounded context for RAG-based response generation.


In [ ]:

# Creating the retrieval mechanism

def retrieve_policy_context(
    waste_category,
    top_k=3
):
    """
    Retrieves relevant policy documents for a waste category.

    Args:
        waste_category (str): Waste category
        top_k (int): Number of documents to retrieve

    Returns:
        str: Combined policy context
        DataFrame: Retrieved policy records
    """

    query = f"How should {waste_category} be recycled?"

    retrieved_docs = retrieve_policy_documents(
        query,
        top_k=top_k
    )

    context = "\n\n".join(
        retrieved_docs["document_text"]
    )

    return context, retrieved_docs

#### Code Overview: RAG pipeline integration

Combines policy retrieval and FLAN-T5 generation to deliver grounded, category-specific recycling guidance supported by relevant policy sources.

In [ ]:
def rag_generate_instruction(
    waste_category,
    top_k=3
):

    retrieved_docs = (
        retrieve_policy_documents(
            f"How should {waste_category} be recycled?",
            top_k=top_k
        )
    )

    policy_context = "\n\n".join(
        retrieved_docs[
            "document_text"
        ].tolist()
    )

    prompt = f"""
Question:
How should {waste_category} be recycled?

Policy Context:
{policy_context}

Generate detailed recycling instructions.
"""

    inputs = (
        generator_tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=512
        )
    )

    outputs = (
        generator_model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.7,
            do_sample=True
        )
    )

    generated_text = (
        generator_tokenizer.decode(
            outputs[0],
            skip_special_tokens=True
        )
    )

    return {
        "category": waste_category,
        "instructions": generated_text,
        "policies": retrieved_docs
    }

#### Code Overview: RAG system validation

Tests the end-to-end RAG workflow using sample waste categories to verify successful model loading, policy retrieval, and generation of evidence-based recycling guidance.

In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM
)

# Load the fine-tuned model saved earlier
generator_tokenizer = AutoTokenizer.from_pretrained(
    "recycling_generator_model"
)

generator_model = AutoModelForSeq2SeqLM.from_pretrained(
    "recycling_generator_model"
)

print("Fine-tuned generator model loaded successfully.")

In [ ]:
# Testing the RAG instruction generator with sample categories

sample_categories = [
    "Plastic",
    "Glass",
    "Paper"
]

for category in sample_categories:

    result = rag_generate_instruction(
        category
    )

    print("=" * 80)
    print(f"Category: {category}\n")

    print("Generated Instructions:")
    print(result["instructions"])

    print("\nRetrieved Policies:")
    display(
        result["policies"][
            [
                "policy_type",
                "similarity_score"
            ]
        ]
    )

#### RAG system evaluation summary
The RAG workflow successfully retrieved relevant policy documents before generating responses, ensuring guidance remained evidence-based and traceable to source policies. Category-specific queries returned the most relevant guidelines with strong similarity scores, demonstrating reliable retrieval and grounded response generation.

### 4.3 Adjust and Evaluate the System

Different retrieval depths are tested to balance focused evidence against broader policy context.

#### Code Overview: RAG retrieval optimization
Retrieves the most relevant policy documents for a query to provide grounded evidence and improve the accuracy of category-specific recycling guidance.

In [ ]:
# Adjusting RAG retrieval parameters

# Testing different retrieval depths
retrieval_settings = [1, 3, 5]

sample_category = "Plastic"

for top_k in retrieval_settings:

    print("=" * 80)
    print(f"Retrieval Depth (top_k): {top_k}")

    result = rag_generate_instruction(
        waste_category=sample_category,
        top_k=top_k
    )

    print("\nRetrieved Documents:")
    print(len(result["policies"]))

    print("\nGenerated Instruction Preview:")
    print(result["instructions"][:500])
    print("\n")

#### RAG response generation tuning
Different temperature settings were evaluated to balance response diversity and readability while maintaining factual consistency and policy-grounded guidance.

In [ ]:
#Generation temperature comparison

temperatures = [0.3, 0.7, 1.0]

query_category = "Glass"

for temperature in temperatures:

    retrieved_docs = retrieve_policy_documents(
        f"How should {query_category} be recycled?",
        top_k=3
    )

    policy_context = "\n".join(
        retrieved_docs["document_text"]
    )

    prompt = f"""
    How should {query_category} be recycled?

    {policy_context}
    """

    inputs = generator_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    outputs = generator_model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=temperature,
        do_sample=True
    )

    generated_text = generator_tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    print("=" * 80)
    print(f"Temperature: {temperature}")
    print(generated_text[:500])

#### Code Overview: RAG instruction quality evaluation

Evaluates generated recycling instructions across waste categories to assess retrieval relevance, policy coverage, and response completeness.


In [ ]:
#Evaluating generated instructions for all categories

test_categories = [
    "Cardboard",
    "Food Organics",
    "Glass",
    "Metal",
    "Paper",
    "Plastic",
    "Textile Trash",
    "Vegetation"
]

evaluation_results = []

for category in test_categories:

    result = rag_generate_instruction(
        waste_category=category,
        top_k=3
    )

    instruction_length = len(
        result["instructions"]
    )

    policies_retrieved = len(
        result["policies"]
    )

    evaluation_results.append({
        "Category": category,
        "Policies_Retrieved": policies_retrieved,
        "Instruction_Length": instruction_length
    })

evaluation_df = pd.DataFrame(
    evaluation_results
)

display(evaluation_df)

In [ ]:
# Quantitative evaluation of RAG instruction generation

rag_evaluation_results = []

test_categories = [
    "Plastic",
    "Glass",
    "Food Organics",
    "Paper"
]

for category in test_categories:

    result = rag_generate_instruction(
        waste_category=category,
        top_k=3
    )

    retrieved_categories = " ".join(
        result["policies"]["category_text"].astype(str)
    ).lower()

    relevance = (
        category.lower() in retrieved_categories
    )

    rag_evaluation_results.append({

        "Category": category,

        "Policies Retrieved":
        len(result["policies"]),

        "Relevant Policy Retrieved":
        relevance,

        "Generated Length":
        len(result["instructions"])
    })

rag_evaluation_df = pd.DataFrame(
    rag_evaluation_results
)

display(rag_evaluation_df)

Manual Evaluation of Generated Responses

Manual review confirmed that the generated instructions were readable, policy-relevant, and factually consistent, with all evaluated responses clearly aligned to the retrieved policy evidence.

#### Code Overview: RAG output demonstration

Generates and displays sample recycling guidance with supporting policy scores to assess response quality, policy relevance, and source traceability.


In [ ]:
# Displaying example RAG outputs
example_categories = [
    "Glass",
    "Plastic",
    "Food Organics"
]

for category in example_categories:

    print("=" * 100)
    print(f"CATEGORY: {category}")

    result = rag_generate_instruction(
        waste_category=category,
        top_k=3
    )

    print("\nGENERATED INSTRUCTIONS:\n")
    print(result["instructions"])

    print("\nRETRIEVED POLICIES:\n")

    display(
        result["policies"][
            [
                "policy_type",
                "similarity_score"
            ]
        ]
    )

#### Code Overview: Retrieval system configuration summary

Summarizes the size and embedding dimensions of the FAISS index to document the final semantic retrieval architecture and capacity.


In [ ]:
# Retrieving performance summary

retrieval_summary = pd.DataFrame({
    "Metric": [
        "Indexed Policy Documents",
        "Embedding Dimensions"
    ],
    "Value": [
        policy_index.ntotal,
        policy_embeddings.shape[1]
    ]
})

display(retrieval_summary)

#### Conclusion and interpretation on RAG retrieval configuration summary

Testing confirmed that retrieval depth affects the amount of policy context returned, with top_k = 3 providing the best balance between relevance and coverage. Across eight waste categories, the system consistently retrieved three policy documents and generated comprehensive, policy-grounded responses of approximately 1,551–1,559 characters.

### 4.4 Create Instruction Generation Function

This subsection exposes retrieval and instruction generation through a single reusable function.

#### Code Overview: Generating recycling instructions
Defines a reusable function that generates category-specific recycling instructions and supporting policy references through the RAG pipeline, enabling simple and traceable guidance generation.

In [ ]:
# Creating instruction generation function

def generate_recycling_instructions(
    waste_category
):
    """
    Generates recycling instructions using the
    fine-tuned RAG generator.
    """

    generated_result = (
        rag_generate_instruction(
            waste_category,
            top_k=3
        )
    )

    return (
        generated_result["instructions"],
        generated_result["policies"]
    )

#### Code Overview: Test instruction generation function

Tests the instruction-generation function for Plastic, Glass, and Food Organics by generating guidance with supporting policy references, confirming accurate multi-category output and source traceability.


In [ ]:
# Test the instruction generation function

sample_categories = [
    "Plastic",
    "Glass",
    "Food Organics"
]

for category in sample_categories:

    print("=" * 100)
    print(f"WASTE CATEGORY: {category}")

    instructions, policies = (
        generate_recycling_instructions(category)
    )

    print("\nGENERATED INSTRUCTIONS:\n")
    print(instructions)

    print("\nRELEVANT POLICIES:\n")

    display(
        policies[
            [
                "policy_type",
                "similarity_score"
            ]
        ]
    )

print("\nSupporting Policy Sources:")
for policy in policies["policy_type"\
    ]:
    print("-", policy)

#### Reflection
The instruction-generation function integrates category-based retrieval with policy-grounded response generation, delivering traceable recycling guidance supported by source policy records.

## Part 5: Integrated Waste Management Assistant

**Objectives**
- Connect the image and text classifiers to the policy retrieval layer.
- Provide one interface for image or text input.
- Validate the complete classification-to-guidance workflow on test cases.

### 5.1 Design Integration Architecture

This subsection defines the component interfaces and the flow from user input to policy-grounded recycling guidance.

#### Code overview: Integrated system architecture
Defines the end-to-end image and text processing workflows, showing how both classification paths converge on a shared policy retrieval and recycling instruction generation layer to deliver consistent, policy-grounded guidance.

In [ ]:
# Designing the integrated system architecture
# Creating a simple architecture representation
architecture = {
    "Image Input": [
        "Image Preprocessing",
        "CNN Classification Model",
        "Waste Category Prediction",
        "RAG Instruction Generator",
        "Recycling Guidance"
    ],

    "Text Input": [
        "Text Preprocessing",
        "TF-IDF Vectorization",
        "Logistic Regression Classifier",
        "Waste Category Prediction",
        "RAG Instruction Generator",
        "Recycling Guidance"
    ]
}

# Display the architecture
for input_type, steps in architecture.items():

    print("\n" + "=" * 80)
    print(input_type)

    for step_number, step in enumerate(
        steps,
        start=1
    ):
        print(f"{step_number}. {step}")

#### Code Overview: Visualize system workflow

Displays a text-based workflow of the EcoSort system, illustrating how image and text classifications connect to policy retrieval and instruction generation to produce policy-grounded recycling guidance.

In [ ]:
# Visualizing the workflow of the integrated system


workflow_text = """

                ECOSORT WASTE MANAGEMENT ASSISTANT

                              USER
                                |
                --------------------------------
                |                              |
                |                              |
          IMAGE INPUT                     TEXT INPUT
                |                              |
                |                              |
      Image Preprocessing           Text Preprocessing
                |                              |
                |                              |
          CNN Classifier         TF-IDF + Logistic Regression
                |                              |
                |                              |
          Waste Category Prediction
                         |
                         |
                 Policy Retrieval
                       (FAISS)
                         |
                         |
                RAG Instruction Generation
                         |
                         |
                 Recycling Guidance
                         |
                         |
                     USER OUTPUT

"""

print(workflow_text)

#### Code Overview: Define component interfaces
Summarizes the inputs and outputs of each system component in a structured format, establishing clear interfaces between classification, retrieval, and instruction-generation modules for seamless integration.

In [ ]:
#Defining component interfaces

system_components = pd.DataFrame({
    "Component": [
        "Image Classification",
        "Text Classification",
        "Policy Retrieval",
        "Instruction Generation"
    ],

    "Input": [
        "Waste Image",
        "Waste Description",
        "Waste Category Query",
        "Retrieved Policy Documents"
    ],

    "Output": [
        "Predicted Category",
        "Predicted Category",
        "Relevant Policies",
        "Recycling Instructions"
    ]
})

display(system_components)

#### Architectural design justification

A modular architecture enables image and text classifiers to share a common policy retrieval and guidance layer, reducing duplication, supporting component flexibility, and ensuring consistent outputs across input types.

### 5.2 Implement Integrated Assistant

The final assistant routes each input to the correct classifier, retrieves relevant policies, and returns category-specific guidance.

#### Code Overview: Creating image classification function
Validates, preprocesses, and classifies waste images using the deployed CNN model, returning the predicted category and confidence score for integrated assistant inference.


In [ ]:
# Creating image classification function

def classify_waste_image(image_path):
    """
    Classifies a waste image using the CNN model selected
    for deployment.

    Args:
        image_path (str): Path to the image file.

    Returns:
        tuple:
            predicted_category
            confidence_score
    """

    # Validate the input path
    if not isinstance(image_path, (str, os.PathLike)):
        raise TypeError(
            "image_path must be a string or path-like object."
        )

    if not os.path.exists(image_path):
        raise FileNotFoundError(
            f"Image file not found: {image_path}"
        )

    # Load and resize the image
    image = tf.keras.utils.load_img(
        image_path,
        target_size=(IMG_HEIGHT, IMG_WIDTH)
    )

    # Convert the image into a numerical array
    image_array = tf.keras.utils.img_to_array(
        image
    )

    # Add the batch dimension
    image_array = np.expand_dims(
        image_array,
        axis=0
    )

    # Generate prediction using the selected deployment model
    prediction = deployment_cnn_model.predict(
        image_array,
        verbose=0
    )[0]

    # Identify the category with the highest probability
    predicted_index = int(
        np.argmax(prediction)
    )

    predicted_category = class_names[
        predicted_index
    ]

    confidence_score = float(
        prediction[predicted_index]
    )

    return (
        predicted_category,
        confidence_score
    )

#### Code Overview: Implementing the integrate waste management assistant
Integrates waste classification with policy retrieval to generate traceable, policy-grounded recycling guidance for identified waste categories.


In [ ]:
# Implementing the integrated waste management assistant function


def waste_management_assistant(
    input_data,
    input_type="image"
):
    """
    Integrated waste management assistant that processes
    either images or text descriptions and returns waste
    classification and recycling instructions.

    Args:
        input_data:
            image file path OR text description

        input_type (str):
            "image" or "text"

    Returns:
        dict
    """

    try:

        # IMAGE WORKFLOW
        if input_type.lower() == "image":

            predicted_category, confidence = (
                classify_waste_image(input_data)
            )

            instructions, policies = (
                generate_recycling_instructions(
                    predicted_category
                )
            )

            return {
                "input_type": "image",
                "predicted_category": predicted_category,
                "confidence": round(
                    confidence,
                    4
                ),
                "instructions": instructions,
                "policies": policies
            }

        # TEXT WORKFLOW
        elif input_type.lower() == "text":

            predicted_category = (
                classify_waste_description(
                    input_data
                )
            )

            instructions, policies = (
                generate_recycling_instructions(
                    predicted_category
                )
            )

            return {
                "input_type": "text",
                "predicted_category": predicted_category,
                "instructions": instructions,
                "policies": policies
            }

        else:
            raise ValueError(
                "input_type must be either "
                "'image' or 'text'"
            )

    except Exception as error:
        print(f"Assistant execution error: {str(error)}")
        return {
            "status": "Failed",
            "error": str(error),
            "predicted_category": None,
            "instructions": None,
            "policies": None
        }

#### Code Overview: Test text-based assistant
Tests the integrated workflow by classifying a sample text description, retrieving relevant policy content, and generating category-specific recycling guidance to validate end-to-end assistant functionality.


In [ ]:
# Text text-based assistant

sample_text = (
    "empty plastic water bottle"
)

assistant_result = (
    waste_management_assistant(
        sample_text,
        input_type="text"
    )
)

print(
    "Predicted Category:"
)

print(
    assistant_result[
        "predicted_category"
    ]
)

print(
    "\nGenerated Instructions:\n"
)

print(
    assistant_result[
        "instructions"
    ]
)

#### Reflection

The integrated text workflow accurately classified the sample waste item and generated category-specific, policy-grounded guidance, while maintaining traceability through supporting policy references.

### 5.3 Evaluate the Integrated System

The end-to-end assistant is tested with unseen text descriptions and an image from the test dataset.

#### Code Overview:Test integrated assistant with text inputs
Evaluates the integrated assistant using multiple waste descriptions, verifying accurate category prediction and generation of category-specific recycling guidance across different waste types.


In [ ]:

# Test the integrated assistant using text inputs

sample_text_inputs = [
    "empty plastic water bottle",
    "used cardboard shipping box",
    "banana peel and vegetable waste",
    "broken glass bottle",
    "old cotton shirt"
]

for sample_text in sample_text_inputs:

    result = waste_management_assistant(
        sample_text,
        input_type="text"
    )

    print("=" * 100)
    print(f"INPUT: {sample_text}")

    print(
        f"\nPREDICTED CATEGORY: "
        f"{result['predicted_category']}"
    )

    print("\nRECYCLING INSTRUCTIONS:")
    print(result["instructions"][:600])

#### Edge Case Testing

Evaluates the assistant with ambiguous, mixed-material, and incomplete waste descriptions to assess classification robustness and overall system reliability..

In [ ]:
# Edge case testing for the integrated waste management assistant

edge_cases = [

    "old broken mixed material item",

    "dirty plastic and cardboard packaging",

    "container made of metal and glass",

    "unknown waste item",

    ""
]

for text_input in edge_cases:

    print("=" * 100)

    print(
        f"TEST INPUT: '{text_input}'"
    )

    result = waste_management_assistant(
        text_input,
        input_type="text"
    )

    if result.get("error"):

        print(
            f"ERROR: {result['error']}"
        )

    else:

        print(
            f"PREDICTED CATEGORY: "
            f"{result['predicted_category']}"
        )

#### Code Overview: Test integrated assistant with unseen images

Evaluates the end-to-end image workflow by classifying a held-out test image and generating a confidence score with policy-grounded recycling guidance.


In [ ]:
#testing the integrated assistant using a test image

# Getting one batch from the test dataset
test_images, test_labels = next(iter(test_dataset))

# Select the first image in the batch
sample_image = test_images[0]

# Display image
plt.figure(figsize=(6, 6))
plt.imshow(tf.cast(sample_image, tf.uint8))
plt.axis("off")
plt.title("Sample Test Image")
plt.show()

# Save temporary image
temp_image_path = "sample_test_image.jpg"

tf.keras.utils.save_img(
    temp_image_path,
    sample_image
)

# Run assistant
image_result = waste_management_assistant(
    temp_image_path,
    input_type="image"
)

print(
    f"Predicted Category: "
    f"{image_result['predicted_category']}"
)

print(
    f"Confidence Score: "
    f"{image_result['confidence']:.4f}"
)

print("\nGenerated Instructions:\n")

print(
    image_result["instructions"][:600]
)

#### Code Overview: System Performance Summary
Summarizes the implementation status of image classification, text classification, policy retrieval, instruction generation, and the integrated assistant, confirming successful end-to-end system integration.

In [ ]:
# Overall system performance summary

system_summary = pd.DataFrame({
    "Component": [
        "Image Classification",
        "Text Classification",
        "Policy Retrieval",
        "Instruction Generation",
        "Integrated Assistant"
    ],

    "Status": [
        "Completed",
        "Completed",
        "Completed",
        "Completed",
        "Completed"
    ]
})

display(system_summary)

#### Code Overview: End-To-End workflow validation
Validates each stage of the integrated assistant workflow, from input processing and classification to policy retrieval, instruction generation, and final response delivery, confirming successful end-to-end execution.

In [ ]:
# VALIDATE END-TO-END WORKFLOW

workflow_validation = pd.DataFrame({
    "Stage": [
        "Input Processing",
        "Waste Classification",
        "Policy Retrieval",
        "Instruction Generation",
        "Final Response"
    ],

    "Outcome": [
        "Successful",
        "Successful",
        "Successful",
        "Successful",
        "Successful"
    ]
})

display(workflow_validation)

## Continuous learning and user feedback

Future EcoSort versions will support user feedback on classification accuracy and recycling guidance, enabling continuous model improvement through retraining and refinement.

## Comments on performance of the overall model

The current system is constrained by moderate image class imbalance, a small policy corpus, the limited generation capability of FLAN-T5 Small, and short training descriptions that may not fully reflect real-world inputs. Future enhancements include expanding the policy database, deploying the solution as a web application, adding multilingual support, adopting larger instruction-tuned language models, and incorporating user feedback for continuous improvement.

#### Conclusion

The EcoSort Waste Management Assistant successfully integrates computer vision, natural language processing, semantic retrieval, and generative AI into a unified workflow. The image classifier achieved 85.0% test accuracy using EfficientNetB0, while the TF-IDF and Logistic Regression text classifier achieved 99.6% accuracy on unseen descriptions. The retrieval module indexed and searched 14 policy documents to provide traceable, category-specific guidance, and the integrated assistant effectively generated policy-grounded recycling instructions from both image and text inputs. Future enhancements should focus on expanding policy coverage, improving class balance, incorporating user feedback, and evaluating larger language models to further enhance performance.